In [1]:
# import libraries

import math
import json
import base64
import cv2
import os
import re
import numpy as np
import pandas as pd
from nltk.metrics.distance import *
from skimage.metrics import structural_similarity as ssim
from bs4 import BeautifulSoup


### SRI for int or float

In [2]:
def sri_int_or_float(original_output, rerun_output, actual_object_type =  None):    
    
    sri_dict = {}
    
    if actual_object_type:
        sri_dict['actual_object_type'] = actual_object_type
    
    if isinstance(original_output, int) and isinstance(rerun_output, int):
        sri_dict['object_type'] = 'int'
    else:
        sri_dict['object_type'] = 'float'
    
    sri_dict['absolute_match'] = str(original_output) == str(rerun_output)
    
    if sri_dict['absolute_match']:
        sri_dict['sri'] = 1
    else:        
        sri_dict['difference_within_tolerance'] = math.isclose(original_output, rerun_output)
        
        if sri_dict['difference_within_tolerance']:
                sri_dict['sri'] = 1
        else:
            sri_dict['absolute_difference'] = abs(original_output - rerun_output)
            
            if original_output != 0:
                sri_dict['relative_difference'] = str(abs(abs(original_output - rerun_output)/original_output) * 100) + '%'
            else:
                sri_dict['relative_difference'] = 'undefined'
        
            sri_dict['sri'] = 0
        
#     print("int or float called!")
#     print(sri_dict)
    
    return sri_dict

### SRI for list or tuple

In [3]:
def sri_list_or_tuple(original_output, rerun_output, actual_object_type = None):

    sri_dict = {}
    
    if actual_object_type:
        sri_dict['actual_object_type'] = actual_object_type    
    
    sri_dict['object_type'] = str(type(original_output))[8: -2]

    
    sri_dict['absolute_match'] = original_output == rerun_output
    
    if sri_dict['absolute_match']:
        sri_dict['sri'] = 1
    else:
        sri_dict['sri'] = 0
        sri_dict['length_match'] = len(original_output) == len(rerun_output)
        
        if sri_dict['length_match']:
            if all(isinstance(x, type(original_output[0])) for x in original_output) and all(isinstance(x, type(rerun_output[0])) for x in rerun_output):
                sri_dict['after_sorting_match'] = sorted(original_output) == sorted(rerun_output)
            
            index_wise_matches = [x == y for (x, y) in zip(original_output, rerun_output)]
            percentage_of_index_wise_matches = index_wise_matches.count(True) / len(index_wise_matches) 
            sri_dict['percentage_of_index_wise_match'] = str(percentage_of_index_wise_matches * 100) + '%'
            sri_dict['sri'] = percentage_of_index_wise_matches
            
        if len(original_output) > 0 and len(rerun_output) > 0:
            is_all_numerics = all([(isinstance(item, int) or isinstance(item, float)) for item in original_output]) and all([(isinstance(item, int) or isinstance(item, float)) for item in rerun_output])
        
            if is_all_numerics:
                sri_dict['max_element_match'] = max(original_output) == max(rerun_output)
                sri_dict['min_element_match'] = min(original_output) == min(rerun_output)
                
            sri_dict['distinct_element_match'] = set(original_output) == set(rerun_output)
            
            if not sri_dict['distinct_element_match']:
                sri_dict['percentage_of_distinct_element_match'] = str(len(set(original_output).intersection(set(rerun_output))) / max(len(set(original_output)), len(set(rerun_output))) * 100) + '%'
            
            if actual_object_type == 'Series':
                sri_dict['sri'] = len(set(original_output).intersection(set(rerun_output))) / max(len(set(original_output)), len(set(rerun_output)))
        
#     print("list or tuple called!")
#     print(sri_dict)
    
    return sri_dict

### SRI for set

In [4]:
def sri_set(original_output, rerun_output):

    sri_dict = {}
    
    sri_dict['object_type'] = 'set'
    
    sri_dict['absolute_match'] = original_output == rerun_output
    
    if sri_dict['absolute_match']:
        sri_dict['sri'] = 1
    else:
        sri_dict['sri'] = 0
        sri_dict['length_match'] = len(original_output) == len(rerun_output)        
            
        if len(original_output) > 0 and len(rerun_output) > 0:
            is_all_numerics = all([(isinstance(item, int) or isinstance(item, float)) for item in original_output]) and all([(isinstance(item, int) or isinstance(item, float)) for item in rerun_output])
        
            if is_all_numerics:
                sri_dict['max_element_match'] = max(original_output) == max(rerun_output)
                sri_dict['min_element_match'] = min(original_output) == min(rerun_output)                
            
            sri_dict['sri'] = len(original_output.intersection(rerun_output)) / max(len(original_output), len(rerun_output))
            sri_dict['percentage_of_elements_match'] = str( sri_dict['sri'] * 100) + '%'
        
#     print("set called!")
#     print(sri_dict)

    return sri_dict

### SRI for dict

In [5]:
def sri_dictionary(original_output, rerun_output, actual_object_type = None):
    
    sri_dict = {}
    
    if actual_object_type:
        sri_dict['actual_object_type'] = actual_object_type    
    
    sri_dict['object_type'] = 'dict'
    
    sri_dict['absolute_match'] = original_output == rerun_output
    
    if sri_dict['absolute_match']:
        sri_dict['sri'] = 1
    else:
        sri_dict['length_match'] = len(original_output) == len(rerun_output)
        sri_dict['sri'] = 0
              
        if len(original_output) > 0 and len(rerun_output) > 0:
            sri_dict['keys_match'] = original_output.keys() == rerun_output.keys()
            
            if not sri_dict['keys_match']:
                sri_dict['percentage_of_original_keys_match'] = str((len(set(original_output.keys()).intersection(set(rerun_output.keys()))) / len(original_output)) * 100) + '%'
            
            matching_items = {key: original_output[key] for key in original_output if key in rerun_output and original_output[key] == rerun_output[key]}
            sri_dict['percentage_of_original_items_match'] = str((len(matching_items) / len(original_output)) * 100) + '%' 
            sri_dict['sri'] = len(matching_items) / max(len(original_output), len(rerun_output))
                            
#     print("dictionary called!")
#     print(sri_dict)

    
    return sri_dict

### SRI for str

In [6]:
def sri_string(original_output, rerun_output, actual_object_type = None):
    
    sri_dict = {}
    
    if actual_object_type:
        sri_dict['actual_object_type'] = actual_object_type    
    
    sri_dict['object_type'] = 'str'
    
    sri_dict['absolute_match'] = original_output == rerun_output
    
    if sri_dict['absolute_match']:
        sri_dict['sri'] = 1
    else:
        sri_dict['sri'] = 0
        sri_dict['length_match'] = len(original_output) == len(rerun_output)
        sri_dict['stripped_match'] = original_output.strip() == rerun_output.strip()
        sri_dict['case_insensitive_match'] = original_output.casefold() == rerun_output.casefold()
        sri_dict['is_substring'] = (original_output in rerun_output) or (rerun_output in original_output)
        
        if sri_dict['stripped_match']:
            sri_dict['sri'] = 1
        
        if sri_dict['sri'] < 1:
            sri_dict['jaro_winkler_similarity'] = jaro_winkler_similarity(original_output, rerun_output)
            sri_dict['sri']  = sri_dict['jaro_winkler_similarity']
        
#     print("string called!")
#     print(sri_dict)
 
    
    return sri_dict

### SRI for Numpy array

In [7]:
def parse_array_output(array_string):
    
    array_string = str(array_string)[1:-1]
    array_string = array_string[1:-1]
    array_string = array_string.replace("\\n', '", "").replace("...,","").replace("array","").strip().replace("\n',","").replace("'","").replace(", dtype=int64","")
    array_string = array_string[1:-1]
    array_list = eval(array_string)
    array = np.array(array_list)
    
    return array

In [8]:
def advanced_parse_numpy_array_quadrants(array_list, edgeitems):

    top_left_subarray = [ ]
    top_right_subarray = [ ]
    bottom_left_subarray = [ ]
    bottom_right_subarray = [ ]

    for item in array_list[:edgeitems]:
        top_left_subarray.append(item[:edgeitems])

    for item in array_list[:edgeitems]:
        top_right_subarray.append(item[edgeitems+1:])

    for item in array_list[edgeitems+1:]:
        bottom_left_subarray.append(item[:edgeitems])

    for item in array_list[edgeitems+1:]:
        bottom_right_subarray.append(item[edgeitems+1:])

    top_left_subarray = np.array(top_left_subarray)
    top_right_subarray = np.array(top_right_subarray)
    bottom_left_subarray = np.array(bottom_left_subarray)
    bottom_right_subarray = np.array(bottom_right_subarray)
    
    return top_left_subarray, top_right_subarray, bottom_left_subarray, bottom_right_subarray

In [9]:
def compare_array_outputs(original_output, rerun_output):    
    
    sri_dict = {}
          
    sri_dict['object_type'] = "Numpy array"
    sri_dict['sri'] = 0
    
    try:
        sri_dict['absolute_match'] = np.array_equal(original_output, rerun_output)

        if sri_dict['absolute_match']:
            sri_dict['sri'] = 1
        else:
            sri_dict['sri'] = 0        
            sri_dict['data_type_match'] = original_output.dtype == rerun_output.dtype        
            sri_dict['dimension_matched'] = original_output.ndim  == rerun_output.ndim
            sri_dict['size_matched'] = original_output.size == rerun_output.size
            sri_dict['shape_matched'] = original_output.shape  == rerun_output.shape

            if shape_matched:
                result_diff = original_output == rerun_output
                percentage_of_elements_matched = result_diff.sum() / original_output.size
                sri_dict['sri'] = percentage_of_elements_matched
                sri_dict['percentage_of_elements_matched'] = str(sri_dict * 100) + "%"

                if data_type_match:
                    percentage_of_elements_matched = np.isclose(original_output, rerun_output, equal_nan = True).sum() / original_output.size
                    sri_dict['sri'] = max(sri_dict['sri'], percentage_of_elements_matched)
                    sri_dict['percentage_of_elements_matched'] = str(sri_dict * 100) + "%"

            else:
                original_output_1d = original_output.flatten()
                rerun_output_1d = rerun_output.flatten()
                sri_dict['percentage_of_index_independent_match'] = str((len(np.intersect1d(original_output_1d, rerun_output_1d)) / max(len(original_output_1d), len (rerun_output_1d)))*100) + "%"
    except:
        pass
#     print("Original output: ", original_output)
#     print("Rerun output: ", rerun_output)
                
    return sri_dict

In [10]:
def compare_array_outputs_advanced(original_array_string, rerun_array_string):
    
    sri_dict = {}
          
    sri_dict['sri'] = 0        
    
    original_row = 0
    original_col = 0
    rerun_row = 0
    rerun_col = 0
    
    original_array_string = str(original_array_string)[1:-1]
    original_array_string = original_array_string[1:-1]
    original_array_string = original_array_string.replace("\\n', '", "").replace("array","")
    original_array_string = original_array_string[1:-1]
    original_array_list = eval(original_array_string)
    
    rerun_array_string = str(rerun_array_string)[1:-1]
    rerun_array_string = rerun_array_string[1:-1]
    rerun_array_string = rerun_array_string.replace("\\n', '", "").replace("array","")
    rerun_array_string = rerun_array_string[1:-1]
    rerun_array_list = eval(rerun_array_string)
    
    
    try:
        if Ellipsis in original_array_list:
            original_row = original_array_list.index(Ellipsis)
    except:
        pass

    try:
        if Ellipsis in original_array_list[0]:
            original_col = original_array_list[0].index(Ellipsis)
    except:
        pass

    try:
        if Ellipsis in rerun_array_list:
            rerun_row = rerun_array_list.index(Ellipsis)
    except:
        pass

    try:
        if Ellipsis in rerun_array_list[0]:
            rerun_col = rerun_array_list[0].index(Ellipsis)
    except:
        pass

    row = min(original_row, rerun_row) 
    col = min(original_col, rerun_col)

    if row == 0:    # means to divide the array into left and right subarrays

        original_left_subarray = []
        original_right_subarray = []

        rerun_left_subarray = []
        rerun_right_subarray = []

        for item in original_array_list:
            original_left_subarray.append(item[:col])  
        for item in original_array_list:
            original_right_subarray.append(item[col+1:])

        for item in rerun_array_list:
            rerun_left_subarray.append(item[:col])  
        for item in rerun_array_list:
            rerun_right_subarray.append(item[rerun_col - col:])       

        original_left_subarray = np.array(original_left_subarray)
        original_right_subarray = np.array(original_right_subarray)

        rerun_left_subarray = np.array(rerun_left_subarray)
        rerun_right_subarray = np.array(rerun_right_subarray)

        if original_col > rerun_col:
            original_left_subarray, rerun_left_subarray = swap_arrays(original_left_subarray, rerun_left_subarray)
            original_right_subarray, rerun_right_subarray = swap_arrays(original_right_subarray, rerun_right_subarray)

        # trim subarrays
        original_left_subarray = original_left_subarray[:, 0 : col]
        rerun_left_subarray = rerun_left_subarray[:, 0 : col]
        original_right_subarray = original_right_subarray[:, - col: ]
        rerun_right_subarray = rerun_right_subarray[:, - col: ]

        left_subarray_reproducibilty_score = compare_array_outputs(original_left_subarray, rerun_left_subarray)
        right_subarray_reproducibilty_score = compare_array_outputs(original_right_subarray, rerun_right_subarray)

        if left_subarray_reproducibilty_score['sri'] > 0 or right_subarray_reproducibilty_score['sri'] > 0:
            harmonic_mean = (2 * left_subarray_reproducibilty_score['sri'] * right_subarray_reproducibilty_score['sri']) / (left_subarray_reproducibilty_score['sri'] + right_subarray_reproducibilty_score['sri'])
            sri_dict['sri'] = harmonic_mean
        else:
            mean = (left_subarray_reproducibilty_score['sri'] + right_subarray_reproducibilty_score['sri']) / 2
            sri_dict['sri'] = mean

    elif col == 0:   # means to divide the array into top and bottom subarrays

        original_top_subarray = []
        original_bottom_subarray = []

        rerun_top_subarray = []
        rerun_bottom_subarray = []

        for item in original_array_list[0: row]:
            original_top_subarray.append(item)  
        for item in rerun_array_list[0: row]:
            rerun_top_subarray.append(item)

        for item in original_array_list[row + 1: ]:
            original_bottom_subarray.append(item)
        for item in rerun_array_list[rerun_row + 1: ]:
            rerun_bottom_subarray.append(item)    

        original_top_subarray = np.array(original_top_subarray)
        original_bottom_subarray = np.array(original_bottom_subarray)

        rerun_top_subarray = np.array(rerun_top_subarray)
        rerun_bottom_subarray = np.array(rerun_bottom_subarray)    

        # trim subarrays
        rerun_bottom_subarray = rerun_bottom_subarray[- row:, :]

        top_subarray_reproducibilty_score = compare_array_outputs(original_top_subarray, rerun_top_subarray)
        bottom_subarray_reproducibilty_score = compare_array_outputs(original_bottom_subarray, rerun_bottom_subarray)    

        if top_subarray_reproducibilty_score['sri'] > 0 or bottom_subarray_reproducibilty_score['sri'] > 0:
            harmonic_mean = (2 * top_subarray_reproducibilty_score['sri'] * bottom_subarray_reproducibilty_score['sri']) / (top_subarray_reproducibilty_score['sri'] + bottom_subarray_reproducibilty_score['sri'])
            sri_dict['sri'] = harmonic_mean
        else:
            mean = (top_subarray_reproducibilty_score['sri'] + bottom_subarray_reproducibilty_score['sri']) / 2
            sri_dict['sri'] = mean       


    else:   # means to divide the array into four quadrants
        edgeitem = min(original_row, rerun_row)

        original_top_left_subarray, original_top_right_subarray, original_bottom_left_subarray, original_bottom_right_subarray = advanced_parse_numpy_array_quadrants(original_array_list, original_row)
        rerun_top_left_subarray, rerun_top_right_subarray, rerun_bottom_left_subarray, rerun_bottom_right_subarray = advanced_parse_numpy_array_quadrants(rerun_array_list, rerun_row)

        if original_row > rerun_row:
            original_top_left_subarray, rerun_top_left_subarray = swap_arrays(original_top_left_subarray, rerun_top_left_subarray)
            original_top_right_subarray, rerun_top_right_subarray = swap_arrays(original_top_right_subarray, rerun_top_right_subarray)
            original_bottom_left_subarray, rerun_bottom_left_subarray = swap_arrays(original_bottom_left_subarray, rerun_bottom_left_subarray)
            original_bottom_right_subarray, rerun_bottom_right_subarray = swap_arrays(original_bottom_right_subarray, rerun_bottom_right_subarray)

        # trim subarrays
        original_top_left_subarray = original_top_left_subarray[0 : edgeitem, 0 : edgeitem]
        rerun_top_left_subarray = rerun_top_left_subarray[0 : edgeitem, 0 : edgeitem]  
        original_top_right_subarray = original_top_right_subarray[0 : edgeitem, 0 : edgeitem]
        rerun_top_right_subarray = rerun_top_right_subarray[0 : edgeitem, (rerun_col - edgeitem): rerun_col]
        original_bottom_left_subarray = original_bottom_left_subarray[0 : edgeitem, 0 : edgeitem]
        rerun_bottom_left_subarray = rerun_bottom_left_subarray[(rerun_row - edgeitem) : rerun_row, 0 : edgeitem]    
        original_bottom_right_subarray = original_bottom_right_subarray[0 : edgeitem, 0 : edgeitem]
        rerun_bottom_right_subarray = rerun_bottom_right_subarray[(rerun_row - edgeitem) : rerun_row, (rerun_col - edgeitem): rerun_col]

        top_left_subarray_reproducibility_score = compare_array_outputs(original_top_left_subarray, rerun_top_left_subarray)
        top_right_subarray_reproducibility_score = compare_array_outputs(original_top_right_subarray, rerun_top_right_subarray)
        bottom_left_subarray_reproducibility_score = compare_array_outputs(original_bottom_left_subarray, rerun_bottom_left_subarray)
        bottom_left_subarray_reproducibility_score = compare_array_outputs(original_bottom_right_subarray, rerun_bottom_right_subarray)

        sri_dict['sri'] = (top_left_subarray_reproducibility_score['sri'] + top_right_subarray_reproducibility_score['sri'] + bottom_left_subarray_reproducibility_score['sri'] + bottom_left_subarray_reproducibility_score['sri']) / 4
    
    return sri_dict

In [11]:
def swap_arrays(array_a, array_b):
    
    array_c = array_a
    array_a = array_b
    array_b = array_c
    
    return array_a, array_b

### SRI for Pandas dataframe

In [12]:
def parse_dataframe_output(dataframe_string):
    
    html_string = ''.join(dataframe_string)  

    soup = BeautifulSoup(html_string, 'html.parser')
    table = soup.find('table')

    dataframe = pd.read_html(str(table))[0]
    
    return dataframe

In [13]:
def compare_dataframe_outputs(original_output, rerun_output):
    
    
    sri_dict = {}
    
    sri_dict['object_type'] = 'Pandas DataFrame'
    
    sri_dict['absolute_match'] = original_output.equals(rerun_output)
    
    if sri_dict['absolute_match']:
        sri_dict['sri'] = 1
    else:
        sri_dict['sri'] = 0
        sri_dict['shape_matched'] = original_output.shape == rerun_output.shape
        sri_dict['columns_matched'] = sorted(list(original_output.columns)) == sorted(list(rerun_output.columns))
        
        if not sri_dict['columns_matched']:
            sri_dict['percentage_of_column_names_matched'] = str((len(set(list(original_output.columns)).intersection(set(list(rerun_output.columns)))) / len(set(list(original_output.columns)))) * 100) + "%"
            sri_dict['percentage_of_index_matched'] = str((len(set(original_output.index).intersection(set(rerun_output.index))) / len(original_output.index)) * 100) + "%"
        
        common_columns = list(dict(set(dict(original_output.dtypes).items()).intersection(set(dict(rerun_output.dtypes).items()))).keys())
        common_indices = list(set(original_output.index).intersection(set(rerun_output.index)))
        
        original_output_to_compare = original_output.filter(items = common_columns, axis = 1)
        original_output_to_compare = original_output_to_compare.filter(items = common_indices, axis = 0)
        
        rerun_output_to_compare = rerun_output.filter(items = common_columns, axis = 1)
        rerun_output_to_compare = rerun_output_to_compare.filter(items = common_indices, axis = 0)
        
        percentage_of_elements_matched = sum(dict(original_output_to_compare.eq(rerun_output_to_compare).sum()).values()) / original_output.size
        sri_dict['percentage_of_elements_matched'] = str(percentage_of_elements_matched * 100) + "%"
        sri_dict['sri'] = percentage_of_elements_matched

#     print("Original output: ", original_output)
#     print("Rerun output: ", rerun_output)
    
        
    return sri_dict

### SRI for images

In [14]:
def sri_image(original_image_file, rerun_image_file):
    
    sri_dict = {}
    
    sri_dict['object_type'] = 'image'    
    
    original_image = cv2.imread(original_image_file)
    rerun_image = cv2.imread(rerun_image_file)

    original_image_output = cv2.cvtColor(original_image, cv2.COLOR_BGR2GRAY)
    rerun_image_output = cv2.cvtColor(rerun_image, cv2.COLOR_BGR2GRAY)    

    rerun_image_output = cv2.resize(rerun_image_output, (original_image_output.shape[1], original_image_output.shape[0]), interpolation = cv2.INTER_CUBIC )                                

    ssim_score = ssim(original_image_output, rerun_image_output)

    sri_dict['sri'] = ssim_score

    
    return sri_dict

### SRI for bool

In [15]:
def sri_bool(original_output, rerun_output):
    
    sri_dict = {}
    
    sri_dict['object_type'] = 'bool'    

    sri_dict['sri'] = int(original_output and rerun_output)

    
    return sri_dict

### Determine Miscellaneous output types

In [16]:
def determine_miscellaneous_output_type(original_misc_output, rerun_misc_output):
    
    datetime_pattern = r'\d{4}-\d{2}-\d{2}[ T]\d{2}:\d{2}:\d{2}'
    if re.search(datetime_pattern, original_misc_output) and re.search(datetime_pattern, rerun_misc_output):
        return "datetime"
   
    memory_address_pattern = r'<.*?object at 0x[0-9a-fA-F]+>'
    if re.search(memory_address_pattern, original_misc_output) and re.search(memory_address_pattern, rerun_misc_output):
        return "memory_address"

    path_pattern = r'([A-Za-z]:\\[^:*?"<>|\r\n]+|\/[^:*?"<>|\r\n]+)'
    if re.search(path_pattern, original_misc_output) and re.search(path_pattern, rerun_misc_output):
        return "path"

    sklearn_model_pattern = r'\b(?:LogisticRegression|RandomForestClassifier|KMeans|DBSCAN|DecisionTreeClassifier|' \
                            r'SVC|LinearRegression|Ridge|Lasso|GradientBoostingClassifier|GaussianNB|' \
                            r'AgglomerativeClustering|MiniBatchKMeans|Birch|MeanShift|' \
                            r'\w+(Classifier|Regression|Clustering))\b\s*\(.*?\)'
    if re.search(sklearn_model_pattern, original_misc_output) and re.search(sklearn_model_pattern, rerun_misc_output):
        return "ml_model"

    return None

### Final Results

In [17]:
def apply_sri(original_notebook, rerun_notebook):    

    notebook = original_notebook.replace("OriginalNotebooks/","")
    
    sri_notebook = dict()  
    sri_cells = list()
    sri_total = 0
    op_compared = 0
    
    
    int_op = 0
    float_op = 0
    list_op = 0
    tuple_op = 0
    set_op = 0
    str_op = 0
    dict_op = 0
    array_op = 0
    dataframe_op = 0

    stream_op = 0
    image_op = 0

    other_op = 0

    int_op_reproduced = 0
    float_op_reproduced = 0
    list_op_reproduced = 0
    tuple_op_reproduced = 0
    set_op_reproduced = 0
    str_op_reproduced = 0
    dict_op_reproduced = 0
    array_op_reproduced = 0
    dataframe_op_reproduced = 0

    stream_op_reproduced = 0
    image_op_reproduced = 0

    other_op_reproduced = 0
    
    int_op_sri = 0
    float_op_sri = 0
    list_op_sri = 0
    tuple_op_sri = 0
    set_op_sri = 0
    str_op_sri = 0
    dict_op_sri = 0
    array_op_sri = 0
    dataframe_op_sri = 0
    
    other_op_sri = 0
    

    sri_notebook['original_nb'] = original_notebook
    sri_notebook['rerun_nb'] = rerun_notebook    

    with open(original_notebook, mode= "r", encoding= "utf-8") as original_file:
        original_nb_json = json.loads(original_file.read())
        
    with open(rerun_notebook, mode= "r", encoding= "utf-8") as rerun_file:
        rerun_nb_json = json.loads(rerun_file.read())
    
    
    for (original_cell, rerun_cell) in zip(original_nb_json['cells'], rerun_nb_json['cells']):            
        if original_cell['cell_type'] == 'code' and rerun_cell['cell_type'] == 'code':

            if len(original_cell['outputs']) > 0 and len(rerun_cell['outputs']) > 0:
                sri_cell = dict()
                sri_cell['original_cell_id'] = original_cell['execution_count']
                sri_cell['rerun_cell_id'] = rerun_cell['execution_count']
                sri_this_cell = list()
                cell_sri_score = 0
                for (original_output, rerun_output) in zip(original_cell['outputs'], rerun_cell['outputs']):                        
                                        
                    if original_output['output_type'] == 'stream' and rerun_output['output_type'] == 'stream':
                                            
                        try:                             
                            sri_str = sri_string(str(original_output['text']), str(rerun_output['text']))
                            sri_this_cell.append(sri_str)
                            cell_sri_score = sri_str['sri']                            
                            sri_total += cell_sri_score
                            op_compared += 1
                            
                            stream_op += 1
                            if cell_sri_score == 1:
                                stream_op_reproduced += 1
                                                                                
                        except:
                            pass
                        

                    if original_output['output_type'] == 'execute_result' and rerun_output['output_type'] == 'execute_result':
                        for (original_data_key, rerun_data_key) in zip(original_output['data'].keys(), rerun_output['data'].keys()):
                            if original_data_key == 'text/plain' and rerun_data_key == 'text/plain':
                                output_type = 'undetermined'
                                try:
                                    type_information_list = rerun_cell['metadata']['type_information']
                                    type_information_list= eval(type_information_list)
                                    rerun_output_type= type_information_list[0]['name']
                                    output_type = rerun_output_type
                                except:
                                    try:
                                        if len(original_output['data'][original_data_key]) == 1:
                                            original_output_type = str(type(eval(original_output['data'][original_data_key][0]))).replace("<class ","").replace(">","").replace("'","")                                        
                                            output_type = original_output_type
                                        else:
                                            output_type = 'undetermined'
                                    except:
                                        output_type = 'undetermined'                                                                                                                                
                                try:
                                    if output_type in ['int', 'int64' 'float', 'float64']:
                                        original_output = eval(original_output['data'][original_data_key][0])
                                        rerun_output = eval(rerun_output['data'][rerun_data_key][0])                                       
                                        sri_int_float_dict = sri_int_or_float(original_output, rerun_output)
                                        sri_this_cell.append(sri_int_float_dict)
                                        cell_sri_score = sri_int_float_dict['sri']
                                        
                                        if output_type in ['int', 'int64']:
                                            int_op += 1
                                            if cell_sri_score == 1:
                                                int_op_reproduced += 1
                                            elif cell_sri_score > 0:
                                                int_op_sri += 1
                                        else:
                                            float_op += 1
                                            if cell_sri_score == 1:
                                                float_op_reproduced += 1 
                                            elif cell_sri_score > 0:
                                                float_op_sri += 1
                                    elif output_type in ['list', 'tuple']:
                                        try:
                                            original_output = eval(str(original_output['data'][original_data_key][0]))
                                            rerun_output = eval(str(rerun_output['data'][rerun_data_key][0])) 
                                            sri_list_tuple = sri_list_or_tuple(original_output, rerun_output)
                                            sri_this_cell.append(sri_list_tuple)
                                            cell_sri_score = sri_list_tuple['sri']
                                        except:
                                            print("!!! Issue with parsing list")
                                            sri_str_dict = sri_string(str(original_output['data'][original_data_key]), str(rerun_output['data'][rerun_data_key]), actual_object_type = 'list')                                    
                                            sri_this_cell.append(sri_str_dict)
                                            cell_sri_score = sri_str_dict['sri']
                                            
                                        if output_type == 'list':
                                            list_op += 1
                                            if cell_sri_score == 1:
                                                list_op_reproduced += 1
                                            elif cell_sri_score > 0:
                                                list_op_sri += 1
                                        else:
                                            tuple_op += 1
                                            if cell_sri_score == 1:
                                                tuple_op_reproduced += 1
                                            elif cell_sri_score > 0:
                                                tuple_op_sri += 1
                                    elif output_type == 'set':
                                        original_output = eval(str(original_output['data'][original_data_key][0]))
                                        rerun_output = eval(str(rerun_output['data'][rerun_data_key][0]))                                        
                                        sri_set_dict = sri_set(original_output, rerun_output)
                                        sri_this_cell.append(sri_set_dict)
                                        cell_sri_score = sri_set_dict['sri']
                                        
                                        set_op += 1
                                        if cell_sri_score == 1:
                                            set_op_reproduced += 1
                                        elif cell_sri_score > 0:
                                            set_op_sri += 1
                                    elif output_type == 'str':
                                        sri_str_dict = sri_string(str(original_output['data'][original_data_key][0]), str(rerun_output['data'][rerun_data_key][0]))                                    
                                        sri_this_cell.append(sri_str_dict)
                                        cell_sri_score = sri_str_dict['sri']
                                        
                                        str_op += 1
                                        if cell_sri_score == 1:
                                            str_op_reproduced += 1
                                        elif cell_sri_score > 0:
                                            str_op_sri += 1
                                        
                                    elif output_type == 'dict':
#                                         print("\n### Dictionary\n")
                                        original_output_list = original_output['data'][original_data_key]                                        
                                        try:
                                            original_output_str = str(original_output_list)[1 : -1]
                                            original_output_str = original_output_str.replace("\\n","")
                                            original_output_str = original_output_str[1:-1]
                                            original_dictionary = eval(original_output_str)

                                            rerun_output_list = rerun_output['data'][rerun_data_key]                                    
                                            rerun_output_str = str(rerun_output_list)[1 : -1]
                                            rerun_output_str = rerun_output_str.replace("\\n","")
                                            rerun_output_str = rerun_output_str[1:-1]
                                            rerun_dictionary = eval(rerun_output_str)
                                            
                                            sri_dictionary_dict = sri_dictionary(original_dictionary, rerun_dictionary)
                                            sri_this_cell.append(sri_dictionary_dict)
                                            cell_sri_score = sri_dictionary_dict['sri']
                                            
                                        except:
                                            sri_str_dict = sri_string(str(original_output['data'][original_data_key][0]), str(rerun_output['data'][rerun_data_key][0]), actual_object_type = 'dict')                                    
                                            sri_this_cell.append(sri_str_dict)
                                            cell_sri_score = sri_str_dict['sri']

                                        dict_op += 1
                                        if cell_sri_score == 1:
                                            dict_op_reproduced += 1
                                        elif cell_sri_score > 0:
                                            dict_op_sri += 1
                                    elif output_type == 'ndarray':
#                                         print("\n### Numpy array \n")
                                        original_output_string = original_output['data'][original_data_key]
                                        rerun_output_string = rerun_output['data'][rerun_data_key]
                                        
                                        if '...' in str(original_output_string) and '...' in str(rerun_output_string):                                            
                                            try:
                                                sri_array_dict = compare_array_outputs_advanced(original_output_string, rerun_output_string)
                                                sri_this_cell.append(sri_array_dict)
                                                cell_sri_score = sri_array_dict['sri']
                                            except:
                                                print("!!! Issue with parsing Numpy array")
                                        
                                        try:
                                            original_array = parse_array_output(original_output_string)
                                            rerun_array = parse_array_output(rerun_output_string)

                                            try:
                                                sri_array_dict = compare_array_outputs(original_array, rerun_array)
                                                sri_this_cell.append(sri_array_dict)
                                                cell_sri_score = sri_array_dict['sri']
                                            except:
                                                sri_str_dict = sri_string(str(original_output['data'][original_data_key]), str(rerun_output['data'][rerun_data_key]), actual_object_type = 'Numpy array')                                    
                                                sri_this_cell.append(sri_str_dict)
                                                cell_sri_score = sri_str_dict['sri']
                                        except:
                                                sri_str_dict = sri_string(str(original_output['data'][original_data_key]), str(rerun_output['data'][rerun_data_key]), actual_object_type = 'Numpy array')                                    
                                                sri_this_cell.append(sri_str_dict)
                                                cell_sri_score = sri_str_dict['sri']
                                                 
                                        array_op += 1
                                        if cell_sri_score == 1:
                                            array_op_reproduced += 1
                                        elif cell_sri_score > 0:
                                            array_op_sri += 1
                                    elif output_type == 'DataFrame':
#                                         print("\n### Pandas dataframe\n")
                                        try:
                                            original_output_text = original_output['data']['text/html']
                                            rerun_output_text = rerun_output['data']['text/html']                                            

                                            original_dataframe = parse_dataframe_output(original_output_text)
                                            rerun_dataframe = parse_dataframe_output(rerun_output_text)

                                            sri_df_dict = compare_dataframe_outputs(original_dataframe, rerun_dataframe)
                                            sri_this_cell.append(sri_df_dict)
                                            cell_sri_score = sri_df_dict['sri']
                                        except:
                                            sri_str_dict = sri_string(str(original_output['data'][original_data_key]), str(rerun_output['data'][rerun_data_key]), actual_object_type = 'Pandas DataFrame')                                    
                                            sri_this_cell.append(sri_str_dict)
                                            cell_sri_score = sri_str_dict['sri']
                                        
                                        dataframe_op += 1
                                        if cell_sri_score == 1:
                                            dataframe_op_reproduced += 1
                                        elif cell_sri_score > 0:
                                            dataframe_op_sri += 1
                                    elif output_type == 'dict_keys':
                                        try:                                            
                                            original_output = str(str(original_output['data'][original_data_key][0]).replace("dict_keys","")[1:-1])
                                            rerun_output = str(str(rerun_output['data'][rerun_data_key][0]).replace("dict_keys","")[1:-1])
                                            original_output = eval(original_output)
                                            rerun_output = eval(rerun_output)
                                            sri_list_tuple = sri_list_or_tuple(original_output, rerun_output, actual_object_type = 'dict_keys')
                                            sri_this_cell.append(sri_list_tuple)
                                            cell_sri_score = sri_list_tuple['sri']
                                        except:
                                            sri_str_dict = sri_string(str(original_output['data'][original_data_key]), str(rerun_output['data'][rerun_data_key]), actual_object_type = 'dict_keys')                                    
                                            sri_this_cell.append(sri_str_dict)
                                            cell_sri_score = sri_str_dict['sri']

                                        other_op += 1
                                        if cell_sri_score == 1:
                                            other_op_reproduced += 1
                                        elif cell_sri_score > 0:
                                            other_op_sri += 1
                                    elif output_type == 'Series':
                                        try:
                                            original_output = original_output['data'][original_data_key]
                                            rerun_output = rerun_output['data'][rerun_data_key]
                                            sri_list_tuple = sri_list_or_tuple(original_output, rerun_output, actual_object_type = 'Series')
                                            sri_this_cell.append(sri_list_tuple)
                                            cell_sri_score = sri_list_tuple['sri']
                                        except:
                                            sri_str_dict = sri_string(str(original_output['data'][original_data_key]), str(rerun_output['data'][rerun_data_key]), actual_object_type = 'Series')                                    
                                            sri_this_cell.append(sri_str_dict)
                                            cell_sri_score = sri_str_dict['sri']
                                        
                                        other_op += 1
                                        if cell_sri_score == 1:
                                            other_op_reproduced += 1
                                        elif cell_sri_score > 0:
                                            other_op_sri += 1
                                    elif output_type == 'bool':
                                            sri_str_bool = sri_bool(str(original_output['data'][original_data_key]), str(rerun_output['data'][rerun_data_key]))                                    
                                            sri_this_cell.append(sri_str_bool)
                                            cell_sri_score = sri_str_bool['sri']

                                            other_op += 1
                                            if cell_sri_score == 1:
                                                other_op_reproduced += 1

                                    elif output_type == 'datetime':
                                        sri_str_dict = sri_string(str(original_output['data'][original_data_key][0]), str(rerun_output['data'][rerun_data_key][0]), actual_object_type = 'datetime')                                    
                                        sri_this_cell.append(sri_str_dict)
                                        cell_sri_score = sri_str_dict['sri']
                                        
                                        other_op += 1
                                        if cell_sri_score == 1:
                                            other_op_reproduced += 1
                                        elif cell_sri_score > 0:
                                            other_op_sri += 1

                                        cell_sri_score = 0 # to avoid adding to the overall sri score
                                        op_compared -= 1 # to balance the count of output compared 

                                    else:
                                        misc_type = determine_miscellaneous_output_type(str(original_output['data'][original_data_key][0]), str(rerun_output['data'][rerun_data_key][0]))
                                        
                                        if misc_type:
                                            sri_str_dict = sri_string(str(original_output['data'][original_data_key][0]), str(rerun_output['data'][rerun_data_key][0]), actual_object_type = misc_type)                                    
                                            sri_this_cell.append(sri_str_dict)
                                            cell_sri_score = sri_str_dict['sri']
                                            
                                            other_op += 1
                                            if cell_sri_score == 1:
                                                other_op_reproduced += 1
                                            elif cell_sri_score > 0:
                                                other_op_sri += 1
                                            
                                            cell_sri_score = 0 # to avoid adding to the overall sri score
                                            op_compared -= 1 # to balance the count of output compared
                                        else:                                        
                                            sri_str_dict = sri_string(str(original_output['data'][original_data_key][0]), str(rerun_output['data'][rerun_data_key][0]))                                    
                                            sri_this_cell.append(sri_str_dict)
                                            cell_sri_score = sri_str_dict['sri']
                                                    
                                    sri_total += cell_sri_score
                                    op_compared += 1

                                except:
                                    pass
                
                    elif original_output['output_type'] == 'display_data' and rerun_output['output_type'] == 'display_data':
                        for (original_data_key, rerun_data_key) in zip(original_output['data'].keys(), rerun_output['data'].keys()):
                            if original_data_key == 'image/png' and rerun_data_key == 'image/png':
                                byte_string = bytes(original_output['data'][original_data_key], 'utf-8')
                                with open("original_image_output.png", "wb") as f:
                                    f.write(base64.decodebytes(byte_string))                                    
                                byte_string = bytes(rerun_output['data'][rerun_data_key], 'utf-8')
                                with open("rerun_image_output.png","wb") as f:
                                    f.write(base64.decodebytes(byte_string))
                                
                                sri_img = sri_image("original_image_output.png", "rerun_image_output.png")
                                sri_this_cell.append(sri_img)
                                cell_sri_score = sri_img['sri']
                                
                                sri_total += cell_sri_score
                                op_compared += 1
                                image_op += 1
                                if cell_sri_score == 1:
                                    image_op_reproduced += 1
                                os.remove("original_image_output.png")
                                os.remove("rerun_image_output.png")

                sri_cell['sri_this_cell'] = sri_this_cell
                sri_cells.append(sri_cell)
                
    
    sri_notebook['sri_cells'] = sri_cells
    if op_compared != 0:
        sri_notebook['sri_avg'] = sri_total / op_compared
    else: 
        sri_notebook['sri_avg'] = 0    
    
    notebook_sri_score = sri_notebook['sri_avg']
            
    json_data = sri_notebook
    json_file_name = "JSONResults/" + notebook.replace("OriginalNotebooks/","") +'.json'

    with open(json_file_name, 'w') as json_file:
        json.dump(json_data, json_file, indent = 4)
    
    
    return [notebook,notebook_sri_score,int_op,int_op_reproduced,int_op_sri,float_op,float_op_reproduced,float_op_sri,list_op,list_op_reproduced,list_op_sri,tuple_op,tuple_op_reproduced,tuple_op_sri,set_op,set_op_reproduced,set_op_sri,str_op,str_op_reproduced,str_op_sri,dict_op,dict_op_reproduced,dict_op_sri,array_op,array_op_reproduced,array_op_sri,dataframe_op,dataframe_op_reproduced,dataframe_op_sri,other_op,other_op_reproduced,other_op_sri,image_op,image_op_reproduced,stream_op,stream_op_reproduced]    

In [ ]:
list_of_headers = ["notebook", "notebook_sri_score", "int_op", "int_op_reproduced", "int_op_sri",
 "float_op", "float_op_reproduced", "float_op_sri", "list_op", "list_op_reproduced",
 "list_op_sri", "tuple_op", "tuple_op_reproduced", "tuple_op_sri", "set_op",
 "set_op_reproduced", "set_op_sri", "str_op", "str_op_reproduced", "str_op_sri",
 "dict_op", "dict_op_reproduced", "dict_op_sri", "array_op", "array_op_reproduced",
 "array_op_sri", "dataframe_op", "dataframe_op_reproduced", "dataframe_op_sri",
 "other_op", "other_op_reproduced", "other_op_sri", "image_op", "image_op_reproduced",
 "stream_op", "stream_op_reproduced"]

results = pd.DataFrame(columns = list_of_headers)

notebooks = pd.read_csv("100_notebooks.csv")['notebooks'].tolist()

for notebook in notebooks:    
    
#     print("Notebook: ", notebook)
    
    original_notebook = "OriginalNotebooks/" + notebook
    rerun_notebook = "RerunNotebooks/" + notebook

    results.loc[len(results)] = apply_sri(original_notebook, rerun_notebook)

#     print("\n\n")

In [48]:
results

,notebook,notebook_sri_score,int_op,int_op_reproduced,int_op_sri,float_op,float_op_reproduced,float_op_sri,list_op,list_op_reproduced,...,dataframe_op,dataframe_op_reproduced,dataframe_op_sri,other_op,other_op_reproduced,other_op_sri,image_op,image_op_reproduced,stream_op,stream_op_reproduced
0,f40b1012ac53f001adc118bbf9dd4d656c9a5d83.ipynb,0.999687,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,28,27
1,bd2e5bfd512e757eeb8c3ad0f375d2949efef8e5.ipynb,0.000000,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,fcce93e9e3c37043188e02cbb01d26c28a76206f.ipynb,0.889532,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,11,11
3,e90e4de2a9244a26e96cdbb1163950a90d2a5827.ipynb,0.708272,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
4,01b90cdc06c9c946a090bdc853ca0d1696c9f343.ipynb,0.925220,0,0,0,0,0,0,0,0,...,0,0,0,1,1,0,3,0,12,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,defc752334f2fef4201f84f23eac29456a5749fa.ipynb,0.916806,5,4,0,0,0,0,2,2,...,0,0,0,0,0,0,0,0,11,9
96,1717286d6b59f4a3b2f2b0e6bf65b0a2d2c689a4.ipynb,1.000000,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
97,b17dcc5096c9dd8d31178a0f5b03f1de92da5a5a.ipynb,1.000000,1,1,0,0,0,0,0,0,...,4,4,0,2,2,0,0,0,3,3
98,8cca42209edf73c9740eb049dee0c0b942dca295.ipynb,0.988627,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,15,14


In [49]:
results.to_csv("sri_final_results.csv", index = False)